# Delta Hedging & Adversarial Stock Path Generation Simulator

An end-to-end quantitative finance and deep learning environment for simulating options markets, generating synthetic paths under multiple market regimes (GBM, Stress/Jump Diffusion, Adversarial Scenarios), and training neural network hedgers under proportional transaction costs.

### Table of Contents
1. **Market Data & Greeks Calibration** (Calibrated from real NSE NIFTY Options & Futures)
2. **3 Stock Path Generation Techniques** (GBM, Jump Diffusion, Adversarial Stress Paths)
3. **Discrete-Time Delta Hedging Engine** (Transaction Cost Modeling & Cashflow Accounting)
4. **PyTorch Deep Hedger Training** (Differentiable CVaR / Expected Shortfall Loss Optimization)
5. **Out-of-Sample Performance Benchmarking** (P&L Distributions, VaR, CVaR, Turnover)

In [ ]:
import os
import sys
# Add project root to path
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.black_scholes import black_scholes_price, black_scholes_greeks, implied_volatility
from src.market_data import load_nifty_minute_data, get_options_snapshot
from src.path_generators import generate_gbm_paths, generate_stress_jump_paths, generate_adversarial_paths
from src.hedging_engine import compute_black_scholes_deltas, simulate_hedging_pnl
from src.deep_hedger import DeepHedger
from src.metrics import evaluate_hedging_performance, compare_strategies
from src.visualizer import plot_pnl_comparison, plot_sample_paths

print('Libraries and custom modules loaded successfully!')

## 1. Load Real NSE NIFTY Minute Options Data
We calibrate market parameters ($S_0, K, \sigma_{\text{implied}}, r, T$) directly from real NIFTY option chain data.

In [ ]:
csv_path = '../drive-download-20260903T145837Z-1-001/20260204_option_minute_prices_non_expiry.csv'
df_market = load_nifty_minute_data(csv_path)
spot_price, options_snapshot = get_options_snapshot(df_market, minute_str='110000', r=0.07, days_to_expiry=1.0)

print(f'Calibrated NIFTY Futures Spot Price: {spot_price:.2f} INR')
display(options_snapshot[['symbol', 'strike', 'type', 'price', 'implied_vol', 'bs_delta', 'bs_gamma']].head(10))

## 2. Generate Stock Price Paths across 3 Techniques
We simulate 10,000 Monte Carlo paths using:
1. **Geometric Brownian Motion (GBM)**
2. **Stress-Based Jump Diffusion & Volatility Bursts**
3. **Adversarial Market Scenarios** (pinning whipsaws near strike $K$)

In [ ]:
S0 = spot_price
K = 25750.0  # ATM Call Strike
sigma = 0.15
r = 0.07
T = 1.0 / 252.0  # 1 Trading Day Horizon
num_steps = 100
num_paths = 10000

gbm_paths = generate_gbm_paths(S0, mu=r, sigma=sigma, T=T, num_steps=num_steps, num_paths=num_paths, seed=42)
stress_paths = generate_stress_jump_paths(S0, mu=r, base_sigma=sigma, T=T, num_steps=num_steps, num_paths=num_paths, jump_intensity=5.0, seed=999)
adv_paths = generate_adversarial_paths(S0, K=K, base_sigma=sigma, T=T, num_steps=num_steps, num_paths=num_paths, seed=777)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
t_steps = np.linspace(0, 1, num_steps + 1)
for i in range(10):
    axes[0].plot(t_steps, gbm_paths[i], alpha=0.7)
    axes[1].plot(t_steps, stress_paths[i], alpha=0.7)
    axes[2].plot(t_steps, adv_paths[i], alpha=0.7)

axes[0].set_title('1. Geometric Brownian Motion (GBM)')
axes[1].set_title('2. Stress-Based Jump Diffusion')
axes[2].set_title('3. Adversarial Market Scenarios')
for ax in axes:
    ax.axhline(K, color='black', linestyle='--', label=f'Strike K={K}')
    ax.set_xlabel('Normalized Time')
    ax.set_ylabel('Stock Price')
plt.tight_layout()
plt.show()

## 3. Train Neural Network Deep Hedger
We train a PyTorch neural network to output optimal dynamic delta positions while minimizing **Conditional Value at Risk (CVaR at 5%)** under proportional transaction costs ($c = 0.10\%$).

In [ ]:
hedger = DeepHedger(K=K, T=T, r=r, sigma=sigma, cost_rate=0.001, hidden_dim=32, lr=0.005, alpha=0.05)
loss_hist = hedger.fit(gbm_paths[:8000], epochs=30, batch_size=256, verbose=True)

plt.figure(figsize=(8, 4))
plt.plot(loss_hist, color='navy', lw=2)
plt.title('CVaR Loss Convergence during Deep Hedger Training')
plt.xlabel('Epoch')
plt.ylabel('CVaR (5%) Loss')
plt.grid(True)
plt.show()

## 4. Benchmark Strategies & Evaluate Quantitative Metrics
We evaluate Black-Scholes Delta Hedging vs. the trained Deep Hedger across standard and stressed market regimes.

In [ ]:
test_gbm = gbm_paths[8000:]
bs_deltas = compute_black_scholes_deltas(test_gbm, K, T, r, sigma, 'call')
dh_deltas = hedger.predict_deltas(test_gbm)

pnl_bs = simulate_hedging_pnl(test_gbm, bs_deltas, K, T, r, cost_rate=0.001, sigma_for_premium=sigma)
pnl_dh = simulate_hedging_pnl(test_gbm, dh_deltas, K, T, r, cost_rate=0.001, sigma_for_premium=sigma)

res_bs = evaluate_hedging_performance(pnl_bs, bs_deltas, 'Black-Scholes (10 bps Cost)')
res_dh = evaluate_hedging_performance(pnl_dh, dh_deltas, 'Deep Hedger (10 bps Cost)')

df_comp = compare_strategies([res_bs, res_dh])
display(df_comp)

In [ ]:
plot_pnl_comparison(
    pnl_bs['net_pnl'], pnl_dh['net_pnl'], alpha=0.05,
    title='Out-of-Sample P&L: Black-Scholes vs. Deep Hedger'
);